# 03 — Transformation test

Exercises `src/run_transformations.py`, which calls the master stored
procedure `automation.sp_transform_all()`. This truncates and rebuilds
every `harmonized.*` table from `raw.*` (cleaning, typing, dedup), then
the `analytics.*` views the Streamlit dashboard reads from.

Run `02_ingestion_test.ipynb` first so `raw.*` has data to transform.

In [1]:
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from db_connection import get_engine
from run_transformations import run_transformations

engine = get_engine()

2026-09-22 16:06:02,656 | INFO | db_connection | Database environment variables validated successfully.


2026-09-22 16:06:02,658 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=acnh_analytics, user=postgres


2026-09-22 16:06:02,658 | INFO | db_connection | Creating SQLAlchemy engine.


2026-09-22 16:06:02,713 | INFO | db_connection | Database environment variables validated successfully.


2026-09-22 16:06:02,714 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=acnh_analytics, user=postgres


2026-09-22 16:06:02,714 | INFO | db_connection | Creating SQLAlchemy engine.


In [2]:
run_transformations()

2026-09-22 16:06:02,721 | INFO | run_transformations | Starting transformation process...


2026-09-22 16:06:02,849 | INFO | run_transformations | Transformation process completed successfully.


## Verify: row counts in `harmonized.*`

Base tables should have fewer (or equal) rows than `raw.*`: dedup keeps
only the latest version of each `unique_entry_id`. `housewares` drops the
most, since it also collapses color/pattern variants down to one row per
item (see `sql/04`'s `item_dedup` CTE).

In [3]:
harmonized_tables = [
    "fish", "insects", "fossils", "villagers", "housewares", "recipes",
    "creatures_availability", "creatures_hourly",
    "villager_furniture", "recipe_materials",
]

{
    table: pd.read_sql(f"SELECT COUNT(*) AS n FROM harmonized.{table}", engine)["n"].iloc[0]
    for table in harmonized_tables
}

{'fish': np.int64(26),
 'insects': np.int64(26),
 'fossils': np.int64(24),
 'villagers': np.int64(130),
 'housewares': np.int64(390),
 'recipes': np.int64(198),
 'creatures_availability': np.int64(662),
 'creatures_hourly': np.int64(11132),
 'villager_furniture': np.int64(1578),
 'recipe_materials': np.int64(394)}

## Business question: top 10 most profitable creatures (Northern Hemisphere, January)

This is the query behind the dashboard's main chart — same view
(`analytics.v_top_creatures_by_month`) `app.py` reads from.

In [4]:
pd.read_sql(
    """
    SELECT rank, creature_type, name, sell, time_window
    FROM analytics.v_top_creatures_by_month
    WHERE hemisphere = 'NH' AND month = 1
    ORDER BY rank
    LIMIT 10
    """,
    engine,
)

,rank,creature_type,name,sell,time_window
0,1,fish,Coelacanth,15000,All day
1,1,fish,Barreleye,15000,9 PM – 4 AM
2,3,fish,Sturgeon,10000,All day
3,3,fish,Blue Marlin,10000,All day
4,5,insect,Tarantula,8000,7 PM – 4 AM
5,6,fish,Blowfish,5000,9 PM – 4 AM
6,7,fish,Goldfish,1300,All day
7,8,insect,Hermit Crab,1000,7 PM – 8 AM
8,8,fish,Sea Butterfly,1000,All day
9,10,insect,Spider,600,7 PM – 8 AM


## Monthly Bell potential (fish vs. insects, Northern Hemisphere)

Backs the dashboard's 12-month stacked bar chart.

In [5]:
monthly = pd.read_sql(
    """
    SELECT month, creature_type, species_count, total_bells
    FROM analytics.v_monthly_bell_potential
    WHERE hemisphere = 'NH'
    ORDER BY month, creature_type
    """,
    engine,
)
monthly.pivot(index="month", columns="creature_type", values="total_bells")

creature_type,fish,insect
month,,
1,58590,10610
2,58590,10610
3,73590,11240
4,64890,18640
5,63190,18590
6,73690,26790
7,88590,41790
8,92340,41990
9,112040,21960
